# MoneyPrinterTurbo Setup Guide

This notebook will guide you through the process of setting up [MoneyPrinterTurbo](https://github.com/xiaosheng1126/MoneyPrinterTurbo) with persistent Google Drive storage.

## 1. Clone Repository and Install Dependencies

First, we'll clone the repository from GitHub and install all required packages:

In [ ]:
%cd /content
!test -d MoneyPrinterTurbo/.git || git clone https://github.com/xiaosheng1126/MoneyPrinterTurbo.git
%cd MoneyPrinterTurbo
!pip install -q uv pyngrok toml
!uv python install 3.11
# Colab 预装了 google-colab / google-genai / google-adk 等包，直接 pip install -r requirements.txt
# 会覆盖全局依赖并触发版本冲突。这里使用 uv 为项目创建独立 .venv，避免影响 Colab 运行时。
!uv sync --frozen --python 3.11

## 2. Mount Google Drive and Persist Configuration

`config.toml` and generated task outputs are stored in `MyDrive/MoneyPrinterTurbo/`, so WebUI changes survive Colab runtime resets.

On first run, non-sensitive defaults come from `CONFIG_OVERRIDES`. Empty API key fields can be initialized from private Colab Secrets using `SECRET_CONFIG_KEYS`. Existing non-empty Drive settings are never overwritten.

In [ ]:
from google.colab import drive
from google.colab import userdata
from pathlib import Path
import toml

project_root = Path("/content/MoneyPrinterTurbo")
drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/MoneyPrinterTurbo")
drive_config = drive_root / "config.toml"
drive_videos = drive_root / "videos"
drive_root.mkdir(parents=True, exist_ok=True)
drive_videos.mkdir(parents=True, exist_ok=True)

# These defaults are applied only when the Drive config is first created.
CONFIG_OVERRIDES = {
    "llm_provider": "openai",
    "openai_model_name": "gpt-4o-mini",
    "openai_base_url": "",
    "video_source": "pexels",
}

# Map config.toml fields to private Colab Secret names.
SECRET_CONFIG_KEYS = {
    "openai_api_key": "OPENAI_API_KEY",
    "aihubmix_api_key": "AIHUBMIX_API_KEY",
    "gemini_api_key": "GEMINI_API_KEY",
    "deepseek_api_key": "DEEPSEEK_API_KEY",
    "pexels_api_keys": "PEXELS_API_KEY",
    "pixabay_api_keys": "PIXABAY_API_KEY",
    "coverr_api_keys": "COVERR_API_KEY",
}

is_new_config = not drive_config.exists()
config_data = toml.load(
    project_root / "config.example.toml" if is_new_config else drive_config
)
app_config = config_data.setdefault("app", {})
if is_new_config:
    app_config.update(CONFIG_OVERRIDES)
for config_key, secret_name in SECRET_CONFIG_KEYS.items():
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if secret_value and not app_config.get(config_key):
        app_config[config_key] = [secret_value] if config_key.endswith("_api_keys") else secret_value

with drive_config.open("w", encoding="utf-8") as config_file:
    toml.dump(config_data, config_file)

def ensure_symlink(link: Path, target: Path):
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"{link} already points to a different location")
        return
    if link.exists():
        raise RuntimeError(f"{link} already exists; move it before rerunning this cell")
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(target, target_is_directory=target.is_dir())

local_config = project_root / "config.toml"
if local_config.exists() or local_config.is_symlink():
    local_config.unlink()
ensure_symlink(local_config, drive_config)
ensure_symlink(project_root / "storage" / "tasks", drive_videos)

print(f"Persistent config: {drive_config}")
print(f"Generated videos: {drive_videos}")

## 3. Configure ngrok for Remote Access

We'll use ngrok to create a secure tunnel to expose our local Streamlit server to the internet.

Create a private Colab secret named `NGROK_AUTH_TOKEN` using the key icon in the left sidebar. Get the token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.kill()
try:
    ngrok_token = userdata.get("NGROK_AUTH_TOKEN")
except userdata.SecretNotFoundError as exc:
    raise RuntimeError(
        "Create a Colab Secret named NGROK_AUTH_TOKEN, enable notebook access, then rerun this cell."
    ) from exc
except userdata.NotebookAccessError as exc:
    raise RuntimeError(
        "Enable notebook access for the NGROK_AUTH_TOKEN Colab Secret, then rerun this cell."
    ) from exc
if not ngrok_token:
    raise RuntimeError("NGROK_AUTH_TOKEN is empty; update the Colab Secret and rerun this cell.")
ngrok.set_auth_token(ngrok_token)

## 4. Launch Application and Generate Public URL

Now we'll start the Streamlit server and create an ngrok tunnel to make it accessible online:

In [ ]:
import subprocess
import time

print("🚀 Starting MoneyPrinterTurbo...")
# 通过 uv run 进入项目 .venv，确保 Streamlit 使用上一步锁定的依赖版本。
streamlit_proc = subprocess.Popen([
    "uv", "run", "streamlit", "run", "./webui/Main.py", "--server.port=8501", "--server.address=0.0.0.0"
])

# Wait for the server to initialize
time.sleep(5)

print("🌐 Creating ngrok tunnel to expose the MoneyPrinterTurbo...")
public_url = ngrok.connect(8501, bind_tls=True)

print("✅ Deployment complete! Access your MoneyPrinterTurbo at:")
print(public_url)